## Bronze Layer: Automated Ingestion with Auto Loader


In [0]:
%run "../00_Setup_Config/project_config"

In [0]:
import dlt
from pyspark.sql.functions import current_timestamp, col

# Fetch the landing zone path from Pipeline Settings
source_path = spark.conf.get("source_path")

@dlt.table(
    name="events_raw",
    comment="Raw ecommerce events ingested via Auto Loader incrementally",
    table_properties={"quality": "bronze"}
)
def bronze_ingestion():
    return (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        # 1. Incremental Ingestion: Checkpoint tracks which files (Oct/Nov) are processed
        .option("cloudFiles.schemaLocation", "/Volumes/ecommerce_analytics_dev/bronze_layer/checkpoints/schema")
        # 2. Performance & Cost: Enforce types early to prevent schema merge errors later
        .option("cloudFiles.schemaHints", "event_time TIMESTAMP, user_id LONG, price DOUBLE")
        # 3. Quota Management: Process 1 file at a time to stay within memory/core limits
        .option("cloudFiles.maxFilesPerTrigger", 1) 
        # 4. Security: Use Managed File Events for optimized Azure storage access
        .option("cloudFiles.useManagedFileEvents", "true") 
        .load(source_path) 
        # 5. Audit Columns (Lineage & Auditing)
        .select(
            "*", 
            current_timestamp().alias("ingestion_timestamp"),
            col("_metadata.file_path").alias("source_file")
        )
    )

In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name

def add_audit_metadata(df):
    """Appends standardized audit columns to any DataFrame."""
    return df.withColumn("ingestion_timestamp", current_timestamp()) \
             .withColumn("source_metadata", input_file_name())